In [ ]:
import os
import kagglehub
import gc

import pandas                             as pd
import numpy                              as np
import matplotlib.pyplot                  as plt
import seaborn                            as sns

from sklearn.preprocessing                import LabelEncoder, StandardScaler

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import GridSearchCV, PredefinedSplit

from sklearn.metrics import (
    precision_recall_curve, average_precision_score, roc_auc_score,
    roc_curve, f1_score, precision_score, recall_score,accuracy_score,
    confusion_matrix, classification_report)

c:\Users\TAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.environ["KAGGLE_API_TOKEN"] = ""
path = kagglehub.dataset_download("berkanoztas/synthetic-transaction-monitoring-dataset-aml")

In [ ]:
file_path_1 = os.path.join(path, "SAML-D.csv")
data = pd.read_csv(file_path_1)



**Data Preprocessing**

In [ ]:
#Kiểm tra Duplicates
print(data.shape)
data = data.drop_duplicates(keep = 'first')
print(data.shape)

(9504852, 12)
(9504852, 12)


In [ ]:
#Kiểm tra null
nan_counts = data.isna().sum()

if nan_counts.any():
    print("Has nan")
    print(nan_counts[nan_counts > 0])
else:
    print("Not nan")

Not nan


In [ ]:
data.tail()

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
9504847,10:57:01,2023-08-23,2453933570,519744068,2247.25,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Small_Fan_Out
9504848,10:57:06,2023-08-23,9805510177,5416607878,927.18,UK pounds,UK pounds,UK,UK,Debit card,0,Normal_Small_Fan_Out
9504849,10:57:06,2023-08-23,7282330957,2995527149,1455.14,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Small_Fan_Out
9504850,10:57:11,2023-08-23,940337377,4812815165,25995.70,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_In
9504851,10:57:12,2023-08-23,105185176,6824994831,9586.08,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_Out


In [ ]:
# Gộp cột Date và Time thành Timestamp
data['Timestamp'] = pd.to_datetime(data['Date'] + ' ' + data['Time'])

# Xóa cột Date và Time cũ (tuỳ chọn)
data = data.drop(columns=['Date', 'Time'])

data = data.sort_values('Timestamp').reset_index(drop=True)

In [ ]:
data['hour'] = data['Timestamp'].dt.hour
data['day_of_week'] = data['Timestamp'].dt.dayofweek

In [ ]:
data[:3]

,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type,Timestamp,hour,day_of_week
0,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits,2022-10-07 10:35:19,10,4
1,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out,2022-10-07 10:35:20,10,4
2,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Small_Fan_Out,2022-10-07 10:35:20,10,4


# **Engineer tabular features for XGBoost**

In [ ]:
data = data.sort_values('Timestamp').reset_index(drop=True)

In [ ]:
# Unique Account: Bank + Account number -> ID
data['sender_id'] = data['Sender_bank_location'].astype(str) + '_' + data['Sender_account'].astype(str)
data['receiver_id'] = data['Receiver_bank_location'].astype(str) + '_' + data['Receiver_account'].astype(str)

In [ ]:
le_payment = LabelEncoder()

In [ ]:
# Temporal features
data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
data['day_sin'] = np.sin(2 * np.pi * data['day_of_week'] / 7)
data['day_cos'] = np.cos(2 * np.pi * data['day_of_week'] / 7)

# Amount features
data['log_amount'] = np.log1p(data['Amount'])


# Cross currency
data['is_cross_currency'] = (data['Payment_currency'].astype(str) != data['Received_currency'].astype(str)).astype(int)

# Payment format
data['payment_type_enc'] = le_payment.fit_transform(data['Payment_type'].astype(str))

# Bank frequency features (avoid leakage)
data['global_txn_index'] = np.arange(len(data))

# data['from_bank_count_past'] = data.groupby('From Bank').cumcount()
# data['to_bank_count_past'] = data.groupby('To Bank').cumcount()

# data['from_bank_freq_past'] = (
#     data['from_bank_count_past'] / (data['global_txn_index'] + 1)).astype(np.float32)

# data['to_bank_freq_past'] = (
#     data['to_bank_count_past'] / (data['global_txn_index'] + 1)).astype(np.float32)

# # Account-level aggregation features
# ## Account transaction counts
data['sender_tx_count_past'] = data.groupby('sender_id').cumcount().astype(np.float32)
data['receiver_tx_count_past'] = data.groupby('receiver_id').cumcount().astype(np.float32)

## Historical sender amount statistics
sender_group = data.groupby('sender_id')['Amount']

data['sender_amount_sum_past'] = (
    sender_group.cumsum() - data['Amount']).astype(np.float32)
data['sender_tx_count_past_int'] = data.groupby('sender_id').cumcount()
data['sender_avg_amount_past'] = (
    data['sender_amount_sum_past'] / data['sender_tx_count_past_int'].replace(0, np.nan))
data['sender_avg_amount_past'] = data['sender_avg_amount_past'].fillna(0).astype(np.float32)

data['sender_std_amount_past'] = (
    data.groupby('sender_id')['Amount']
    .transform(lambda s: s.expanding().std().shift(1))
    .fillna(0)
    .astype(np.float32))

## Historical receiver amount statistics
receiver_group = data.groupby('receiver_id')['Amount']

data['receiver_amount_sum_past'] = (
    receiver_group.cumsum() - data['Amount']).astype(np.float32)
data['receiver_tx_count_past_int'] = data.groupby('receiver_id').cumcount()
data['receiver_avg_amount_past'] = (
    data['receiver_amount_sum_past'] / data['receiver_tx_count_past_int'].replace(0, np.nan))
data['receiver_avg_amount_past'] = data['receiver_avg_amount_past'].fillna(0).astype(np.float32)

data['receiver_std_amount_past'] = (
    data.groupby('receiver_id')['Amount']
    .transform(lambda s: s.expanding().std().shift(1))
    .fillna(0)
    .astype(np.float32))

## Unique counterparties so far
def cumulative_unique_before_current(series):
    seen = set()
    out = []
    for x in series:
        out.append(len(seen))
        seen.add(x)
    return pd.Series(out, index=series.index)

data['sender_unique_receivers_past'] = (
    data.groupby('sender_id')['receiver_id']
    .transform(cumulative_unique_before_current)
    .astype(np.float32))

data['receiver_unique_senders_past'] = (
    data.groupby('receiver_id')['sender_id']
    .transform(cumulative_unique_before_current)
    .astype(np.float32))

## Interaction count between sender and receiver
pair_key = list(zip(data['sender_id'], data['receiver_id']))
data['pair_key'] = pd.Series(pair_key, index=data.index)

data['interaction_count_past'] = (
    data.groupby('pair_key').cumcount().astype(np.float32))

# **Train / Test Model**

**Train, Val, Test split**

In [ ]:
n = len(data)
train_end = int(0.6 * n)
val_end = int(0.8 * n)

train_df = data.iloc[:train_end].copy()
val_df = data.iloc[train_end:val_end].copy()
test_df = data.iloc[val_end:].copy()

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

print("Train time:", train_df['Timestamp'].min(), "->", train_df['Timestamp'].max())
print("Val time  :", val_df['Timestamp'].min(), "->", val_df['Timestamp'].max())
print("Test time :", test_df['Timestamp'].min(), "->", test_df['Timestamp'].max())

Train: 5702911
Val  : 1900970
Test : 1900971
Train time: 2022-10-07 10:35:19 -> 2023-04-16 21:10:33
Val time  : 2023-04-16 21:10:37 -> 2023-06-19 22:08:51
Test time : 2023-06-19 22:08:52 -> 2023-08-23 10:57:12


In [ ]:
val_df["Is_laundering"].value_counts()

Is_laundering
0    1898982
1       1988
Name: count, dtype: int64

In [ ]:
tabular_features = [
    'hour_sin',
    'hour_cos',
    'day_sin',
    'day_cos',
    'log_amount',
    'is_cross_currency',
    'payment_type_enc',
    # 'from_bank_freq_past',
    # 'to_bank_freq_past',
    'sender_avg_amount_past',
    'sender_std_amount_past',
    'receiver_avg_amount_past',
    'receiver_std_amount_past',
    'sender_unique_receivers_past',
    'receiver_unique_senders_past',
    'interaction_count_past']

In [ ]:
target_col = 'Is_laundering'

X_train = train_df[tabular_features].astype(np.float32)
X_val = val_df[tabular_features].astype(np.float32)
X_test = test_df[tabular_features].astype(np.float32)

y_train = train_df[target_col].astype(int).values
y_val = val_df[target_col].astype(int).values
y_test = test_df[target_col].astype(int).values

In [ ]:
X_train_val = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_train_val = np.concatenate([y_train, y_val])

# Quy ước: -1 là tập Train, 0 là tập Validation
test_fold = np.concatenate([
    np.full(X_train.shape[0], -1), # X_train được đánh dấu -1 (chỉ dùng để train)
    np.full(X_val.shape[0], 0)     # X_val được đánh dấu 0 (chỉ dùng để validate)
])
ps = PredefinedSplit(test_fold)


In [ ]:
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    tree_method='hist')

param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 4, 8, 16],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]}

# 4. Cấu hình GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=ps,           # Dùng PredefinedSplit đã tạo ở trên
    refit=False,     # QUAN TRỌNG: KHÔNG tự động train lại trên tập gộp
    verbose=1,
    n_jobs=1)

grid_search.fit(X_train_val, y_train_val)

best_params_xgb = grid_search.best_params_
print('Tham số tốt nhất:', best_params_xgb)

final_model_xgb = XGBClassifier(
    **best_params_xgb,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42)
final_model_xgb.fit(X_train, y_train)

Fitting 1 folds for each of 144 candidates, totalling 144 fits
Tham số tốt nhất: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 100, 'subsample': 1.0}


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'auc'


In [ ]:
gc.collect()


51

In [ ]:
dt_model = DecisionTreeClassifier(
    random_state=42)

param_grid = {
    'criterion': ['gini', 'entropy'],         # Tiêu chí đo lường độ nhiễu khi chia nhánh
    'max_depth': [4, 8, 12, 16],              # Độ sâu tối đa của cây
    'min_samples_split': [4, 8, 12, 50],      # Số lượng mẫu tối thiểu để tiếp tục chia nhánh
    'min_samples_leaf': [2, 10, 50]           # Số lượng mẫu tối thiểu bắt buộc phải có ở lá cuối cùng
}

grid_search = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=ps,           # Dùng đúng tập Validation
    refit=False,     # KHÔNG tự động train lại trên tập gộp
    verbose=1,
    n_jobs=-1)

grid_search.fit(X_train_val, y_train_val)

best_params_dt = grid_search.best_params_
print(f"Tham số tốt nhất cho Decision Tree: {best_params_dt}")

final_model_dt = DecisionTreeClassifier(
    **best_params_dt,
    random_state=42)
final_model_dt.fit(X_train, y_train)

Bắt đầu tìm kiếm tham số tối ưu cho Decision Tree...
Fitting 1 folds for each of 96 candidates, totalling 96 fits

Tham số tốt nhất tìm được:
{'criterion': 'entropy', 'max_depth': 8, 'min_samples_leaf': 50, 'min_samples_split': 4}
ROC AUC cao nhất trên tập Validation: 0.9933

Huấn luyện mô hình cuối cùng chỉ trên X_train...

ROC AUC TRÊN TẬP TEST: 0.9952


In [ ]:
gc.collect()

In [ ]:
logreg_model = LogisticRegression(
    random_state=42,
    class_weight='balanced', # Tự động phạt nặng nếu đoán sai nhóm Rửa tiền
    max_iter=1000            # Tránh lỗi "ConvergenceWarning" khi dữ liệu quá lớn
)

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],  # Mức độ kiểm soát Overfitting (C càng nhỏ, phạt càng mạnh)
    'solver': ['lbfgs']       # Thuật toán giải chóp tối ưu nhất cho dữ liệu lớn
}

# 4. Cấu hình GridSearchCV
grid_search = GridSearchCV(
    estimator=logreg_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=ps,           # Dùng đúng tập Validation
    refit=False,     # KHÔNG tự động train lại trên tập gộp
    verbose=1,
    n_jobs=-1)

print("Bắt đầu tìm kiếm tham số tối ưu cho Logistic Regression...")
grid_search.fit(X_train_val, y_train_val)

best_params_lr = grid_search.best_params_
print(f"Tham số tốt nhất cho Logistic Regression: {best_params_lr}")

# 5. Huấn luyện mô hình cuối cùng CHỈ TRÊN TẬP TRAIN
final_model_lr = LogisticRegression(
    **best_params_lr,
    random_state=42,
    class_weight='balanced',
    max_iter=1000)
final_model_lr.fit(X_train, y_train)

Bắt đầu tìm kiếm tham số tối ưu cho Logistic Regression...
Fitting 1 folds for each of 5 candidates, totalling 5 fits

Tham số tốt nhất tìm được:
{'C': 0.1, 'solver': 'lbfgs'}
ROC AUC cao nhất trên tập Validation: 0.8398

Huấn luyện mô hình cuối cùng chỉ trên X_train...


c:\Users\TAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



ROC AUC TRÊN TẬP TEST (BASELINE): 0.8452
